In [2]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from natasha import Segmenter, MorphVocab, NewsEmbedding, NewsMorphTagger, Doc

# Лемматизация

In [3]:
df=pd.read_excel('Проверка на дубли (внутри как полное совпадение, так и частичное).xlsx')

In [4]:
df = df.drop(columns=["Номер", "Статус"])
df = df.assign(Предложения=lambda x: (x['Название'] + ' ' + x['Описание']).str.lower())
df=df.drop(columns=['Название', 'Описание'])

In [5]:
orig_df = df['Предложения'].astype(str).tolist()  # Все предложения как есть
orig_df = [s.replace("\n","").replace("\t"," ") for s in orig_df]
orig_df = pd.DataFrame(orig_df)

In [6]:
def split_text(text): return text.split()
orig_df['words'] = orig_df[0].apply(split_text)
df=orig_df.drop(columns=0)

In [7]:
segmenter = Segmenter()
morph_vocab = MorphVocab()
emb = NewsEmbedding()
morph_tagger = NewsMorphTagger(emb)

In [8]:
def lemmatize_words(words):
    doc = Doc(' '.join(words))
    doc.segment(segmenter)
    doc.tag_morph(morph_tagger)
    for token in doc.tokens:
        token.lemmatize(morph_vocab)
    return [token.lemma for token in doc.tokens]

In [9]:
df['lemmatized_words'] = df['words'].apply(lemmatize_words)

In [10]:
STOP = {
    'в', 'на', 'с', 'по', 'к', 'из', 'у', 'от', 'до',
    'за', 'для', 'о', 'об', 'со', 'во', 'не', 'ни', 'под',
    'над', 'перед', 'при', 'через', 'между', "что", "или", '.', 'и'
}

In [11]:
def word_remover(words, stop_words): return [word for word in words if word not in stop_words]

df['filtered_words'] = df['lemmatized_words'].apply(lambda words: word_remover(words, STOP))

In [12]:
df.to_csv('after_lemma.csv', index=False, encoding='utf-8')

# Разметка

In [13]:
data = pd.read_csv('after_lemma.csv')

In [14]:
original_data = data['filtered_words']

In [15]:
# 2. Векторизация (без предобработки)
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(original_data)

# 3. Поиск похожих пар
threshold = 0.4
similar_pairs = []
for i in range(len(original_data)):
    for j in range(i+1, len(original_data)):
        sim = cosine_similarity(X[i], X[j])[0][0]
        if sim > threshold:
            similar_pairs.append({
                'ID_1': i,
                'ID_2': j,
                'Сходство': round(sim, 2),
                'Текст_1': original_data[i],
                'Текст_2': original_data[j]
            })

In [ ]:
output_data = []
for idx, text in enumerate(original_data):
    # Находим все пары, где участвует текущее предложение
    related = []
    for pair in similar_pairs:
        if pair['ID_1'] == idx or pair['ID_2'] == idx:
            related_id = pair['ID_2'] if pair['ID_1'] == idx else pair['ID_1']
            related.append(f"{related_id}: {pair['Текст_2'] if pair['ID_1'] == idx else pair['Текст_1']} (сходство: {pair['Сходство']})")
    
    output_data.append({
        'ID': idx,
        'Исходное предложение': text,
        'Похожие предложения': "; ".join(related) if related else "Нет"
    })

df = pd.DataFrame(similar_pairs)
df.to_csv('result.csv', index=False, encoding='utf-8')

In [ ]:
import matplotlib.pyplot as plt

similarities = [pair['Сходство'] for pair in similar_pairs]

plt.hist(similarities, bins=20, edgecolor='black')
plt.xlabel('Сходство')
plt.ylabel('Частота')
plt.title('Распределение сходства между текстами')
plt.show()